In [2]:
#import the library
import scanpy as sc
import numpy as np
import scipy as sp
import pandas as pd
import re
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import colors
import seaborn as sb
from gprofiler import GProfiler
import seaborn as sns
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

import importlib
import warnings
warnings.filterwarnings("ignore")

import pickle as pkl
from matplotlib.colors import LinearSegmentedColormap
import os
os.chdir('E:/AAA_Labwork/T cells')

In [3]:
adata_ab = sc.read_h5ad(r"E:\AAA_Labwork\T cells\v4_revision\revision\250721_gut_liver_TRM_adata_abTCR_conga.h5ad")

In [4]:
adata3 = adata_ab[adata_ab.obs['batch'] == '3',:]
adata4 = adata_ab[adata_ab.obs['batch'] == '4',:]
adata5 = adata_ab[adata_ab.obs['batch'] == '5',:]
adatas=[adata3,adata4,adata5]

In [5]:
cdr3b_nt = pd.Series()
for d in adata_ab.obs['batch'].cat.categories:
    for t in adata_ab.obs['tissue'].cat.categories:
        TCR_info =pd.read_csv('GlobusData/'+t+'/'+d+'/filtered_contig_annotations.csv')
        b_spec = TCR_info[TCR_info['chain']=='TRB']
        
        match_df = adatas[int(d)-3][adatas[int(d)-3].obs['tissue']==t,:].obs[['cdr3b']]
        match_df['barcode'] = [i.split('-1')[0]+'-1' for i in match_df.index]
        
        property_map = b_spec.groupby(['barcode','cdr3'])['cdr3_nt'].apply(set)
        cdr3b_nt = pd.concat([cdr3b_nt,match_df.apply(lambda row: list(property_map[row['barcode'],row['cdr3b']])[0],axis = 1)])
adata_ab.obs['cdr3b_nucseq'] = cdr3b_nt[adata_ab.obs_names]

In [6]:
cdr3a_nt = pd.Series()
for d in adata_ab.obs['batch'].cat.categories:
    for t in adata_ab.obs['tissue'].cat.categories:
        TCR_info =pd.read_csv('GlobusData/'+t+'/'+d+'/filtered_contig_annotations.csv')
        b_spec = TCR_info[TCR_info['chain']=='TRA']
        
        match_df = adatas[int(d)-3][adatas[int(d)-3].obs['tissue']==t,:].obs[['cdr3a']]
        match_df['barcode'] = [i.split('-1')[0]+'-1' for i in match_df.index]
        
        property_map = b_spec.groupby(['barcode','cdr3'])['cdr3_nt'].apply(set)
        cdr3a_nt = pd.concat([cdr3a_nt,match_df.apply(lambda row: list(property_map[row['barcode'],row['cdr3a']])[0],axis = 1)])
adata_ab.obs['cdr3a_nucseq'] = cdr3a_nt[adata_ab.obs_names]

In [7]:
adata_ab.obs['cdr3a_nucseq']

ATCCACCAGCGCCTCA-1-3-LP                  TGCGCCGCGGCGGGAGGAGGAAACAAACTCACCTTT
CCATGTCTCGCCAAAT-1-3-IEL    TGCGCTGTGAGAGATCGGAAACTCACGGGAGGAGGAAACAAACTCA...
ATCTACTCACTTCGAA-1-3-LP         TGCGCTGTGAGACTCTTCACGGGAGGAGGAAACAAACTCACCTTT
GATGAAACACTTCGAA-1-3-LP            TGCGCTGTGACCTCCACGGGAGGAGGAAACAAACTCACCTTT
TTCTCAAAGTCTTGCA-1-4-IEL                       TGCGCTGGGATCTACAGCACCCTCACCTTT
                                                  ...                        
GGCTGGTTCCTTGCCA-1-3-LP            TGTGCTCTGACCCCTAGAACTGGAGGCTTCAAAACTATCTTT
CAGGTGCAGTTGAGAT-1-3-PB            TGTGCTCTGACCCGAAATACTGGAGGCTTCAAAACTATCTTT
GACCTGGTCACAGTAC-1-3-IEL                    TGTGCTCTCACCGTTTTATTCAAAACTATCTTT
TACCTATGTTCCGTCT-1-3-IEL                 TGTGCTCTTTATACTGGAGGCTTCAAAACTATCTTT
GGGACCTAGCTAGTGG-1-5-IEL           TGTGCTCGGTTGGGGGGTTCTGGAGGCTTCAAAACTATCTTT
Name: cdr3a_nucseq, Length: 18806, dtype: object

In [8]:
adata_ab.write(r"E:\AAA_Labwork\T cells\v4_revision\revision\250723_gut_liver_TRM_adata_abTCR_conga.h5ad")